In [1]:
import pandas as pd
import numpy as np
import os
import sys
from matplotlib import pyplot as plt
%matplotlib inline

In [5]:
%cd /content/Incremental-Learning-MycroftMinds-main/notebooks/other_data

/content/Incremental-Learning-MycroftMinds-main/notebooks/other_data


In [6]:
project_root = "Incremental-Learning-MycroftMinds-main"
cwd = os.path.abspath(os.getcwd())
while True:
    if os.path.basename(cwd) == project_root:
        if cwd not in sys.path:
            sys.path.insert(0, cwd)
        break
    parent = os.path.dirname(cwd)
    if parent == cwd:
        raise FileNotFoundError(f"Project root '{project_root}' not found in any parent directory.")
    cwd = parent

In [7]:
from src.data import dataloader
from src.data.io import load_dataset, save_dataset
from src.utils.memory import print_memory_stats
from src.trees import fastodt
from src.data.preprocessing import read_second_batch
import numpy as np
import pandas as pd
import time

### AQ

In [10]:
data_dir = '../../data'

lag_args = {
    'lag_1h':'1H',  # all based on PACF
    'lag_2h':'2H',
    'lag_3h':'3H',
    'lag_24h':'24H',
    #'lag_168h':'168H' # 24h*7 = 168h, one week ago
    }

# list feature names
columns_to_add_lag = [
            'pm2.5'
        ]

# target
target_feature = 'pm2.5'
batch='AQ'
location=None
device_list = None


# Load data using the optimized function
# First try the optimized version, if not available fall back to regular version
train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=device_list,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save=True,  # Save processed data
    intermediate_prefix='sb_AQ_hour_wfv',  # Prefix for saved files
    time_agg='1h',  # Time aggregation frequency
    new_data=batch,
    location=location
)

[INFO] Aggregating data by: 1h
input_features:  ['time', 'device_id', 'DEWP', 'TEMP', 'PRES', 'Iws', 'Is', 'Ir', 'hour', 'weekday', 'month', 'pm2.5_lag_1h', 'pm2.5_lag_2h', 'pm2.5_lag_3h', 'pm2.5_lag_24h']

Processing site: const
Overall data time range:  2010-01-02 01:00:00+01:00 2014-12-31 23:00:00+01:00

Test data time range:  2013-12-31 20:00:00+01:00 2014-12-31 23:00:00+01:00

(Train + Valid) data range:  2010-01-02 01:00:00+01:00 2013-12-31 18:00:00+01:00

Split 1:
  Train window: 2010-01-02 01:00:00+01:00 to 2010-01-30 00:00:00+01:00
  Valid window: 2010-01-30 01:00:00+01:00 to 2010-02-27 00:00:00+01:00
  Train samples: 672, Valid samples: 672

Split 2:
  Train window: 2010-01-02 01:00:00+01:00 to 2010-02-27 00:00:00+01:00
  Valid window: 2010-02-27 01:00:00+01:00 to 2010-03-27 00:00:00+01:00
  Train samples: 1344, Valid samples: 672

Split 3:
  Train window: 2010-01-02 01:00:00+01:00 to 2010-03-27 00:00:00+01:00
  Valid window: 2010-03-27 01:00:00+01:00 to 2010-04-24 01:00:00+0

### Electricity

In [11]:
data_dir = '../../data'

lag_args = {
    'lag_1h':'1H',  # all based on PACF
    'lag_2h':'2H',
    'lag_3h':'3H',
    'lag_24h':'24H',
    #'lag_168h':'168H' # 24h*7 = 168h, one week ago
    }

# list feature names
columns_to_add_lag = [
            'Global_active_power'
        ]

# target
target_feature = 'Global_active_power'
batch='electricity'
location=None
device_list = None


# Load data using the optimized function
# First try the optimized version, if not available fall back to regular version
train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=device_list,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save=True,  # Save processed data
    intermediate_prefix='sb_electricity_hour_wfv',  # Prefix for saved files
    time_agg='1h',  # Time aggregation frequency
    new_data=batch,
    location=location
)

[INFO] Aggregating data by: 1h
input_features:  ['time', 'device_id', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'hour', 'weekday', 'month', 'Global_active_power_lag_1h', 'Global_active_power_lag_2h', 'Global_active_power_lag_3h', 'Global_active_power_lag_24h']

Processing site: const
Overall data time range:  2006-12-17 18:00:00+01:00 2010-11-26 21:00:00+01:00

Test data time range:  2010-02-11 17:00:00+01:00 2010-11-26 21:00:00+01:00

(Train + Valid) data range:  2006-12-17 18:00:00+01:00 2010-02-11 15:00:00+01:00

Split 1:
  Train window: 2006-12-17 18:00:00+01:00 to 2007-01-14 17:00:00+01:00
  Valid window: 2007-01-14 18:00:00+01:00 to 2007-02-11 17:00:00+01:00
  Train samples: 672, Valid samples: 672

Split 2:
  Train window: 2006-12-17 18:00:00+01:00 to 2007-02-11 17:00:00+01:00
  Valid window: 2007-02-11 18:00:00+01:00 to 2007-03-11 17:00:00+01:00
  Train samples: 1344, Valid samples: 672

Split 3:
  Train window

### Friedman

In [ ]:
data_dir = '../../data'

lag_args = {
    'lag_1h':'1H',  # all based on PACF
    'lag_2h':'2H',
    'lag_3h':'3H',
    'lag_24h':'24H',
    #'lag_168h':'168H' # 24h*7 = 168h, one week ago
    }

# list feature names
columns_to_add_lag = [
            'x1', 'x2'
        ]

# target
target_feature = 'y'
batch='friedman'
location=None
device_list = None


# Load data using the optimized function
# First try the optimized version, if not available fall back to regular version
train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=device_list,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save=True,  # Save processed data
    intermediate_prefix='sb_friedman_hour_wfv',  # Prefix for saved files
    time_agg='1h',  # Time aggregation frequency
    new_data=batch,
    location=location
)

[INFO] Aggregating data by: 1h
input_features:  ['time', 'device_id', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'y', 'hour', 'weekday', 'month', 'x1_lag_1h', 'x1_lag_2h', 'x1_lag_3h', 'x1_lag_24h', 'x2_lag_1h', 'x2_lag_2h', 'x2_lag_3h', 'x2_lag_24h']

Processing site: const
Overall data time range:  2000-01-02 01:00:00+01:00 2005-09-14 08:00:00+02:00

Test data time range:  2004-07-24 18:00:00+02:00 2005-09-14 08:00:00+02:00

(Train + Valid) data range:  2000-01-02 01:00:00+01:00 2004-07-24 16:00:00+02:00

Split 1:
  Train window: 2000-01-02 01:00:00+01:00 to 2000-01-30 00:00:00+01:00
  Valid window: 2000-01-30 01:00:00+01:00 to 2000-02-27 00:00:00+01:00
  Train samples: 672, Valid samples: 672

Split 2:
  Train window: 2000-01-02 01:00:00+01:00 to 2000-02-27 00:00:00+01:00
  Valid window: 2000-02-27 01:00:00+01:00 to 2000-03-26 00:00:00+01:00
  Train samples: 1344, Valid samples: 672

Split 3:
  Train window: 2000-01-02 01:00:00+01:00 to 2000-03-26 00:00:00+01:00
  Valid window